Universidad del Valle de Guatemala  
Departamento de Ciencias de la Computación  
CC3084 - Data Science - sección 40

Integrantes:  
- Cristian Túnchez (231359)
- Dulce Ambrosio (231143)
- Daniel Chet (231177)
- Javier Linares (231135)

# 02. Análisis exploratorio de los datos

**MAP - Charting Student Math Misunderstandings**

En este cuaderno se explora el conjunto limpio producido por `01_carga_y_limpieza.ipynb`. El objetivo no es construir un modelo sino entender los insumos: qué forma tienen los datos, qué tan separables son las clases y qué obstáculos concretos condicionarán el modelado.

El trabajo se organiza de la siguiente manera:

1. **Variables cuantitativas.** Estadística descriptiva, histogramas, diagramas de caja y tratamiento de valores atípicos.
2. **Variables categóricas.** Tablas de frecuencia, tablas de proporciones y gráficos de barras.
3. **Cruces entre variables.** Relaciones entre las variables más importantes, correlaciones y medidas de asociación.
4. **Señal léxica.** Qué palabras distinguen a cada categoría.
5. **Hallazgos y conclusiones.**

## 0. Arranque

Este cuaderno funciona igual en **Google Colab** y en una instalación local.

En Colab la celda siguiente clona el repositorio, instala las dependencias y deja el proyecto
listo. Para que la descarga de los datos funcione hay que registrar antes el token de Kaggle
como secreto: en el panel izquierdo de Colab, icono de la llave, agregar un secreto llamado
`KAGGLE_ACCESS_TOKEN` con el contenido de `~/.kaggle/access_token` y habilitar el acceso desde
el cuaderno.

En local la celda no hace nada salvo ubicar la raíz del proyecto.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPOSITORIO = "https://github.com/Tunchxz/math-misconception-map.git"
CARPETA = "math-misconception-map"

if EN_COLAB:
    if not Path(CARPETA).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPOSITORIO, CARPETA], check=True)
    os.chdir(f"/content/{CARPETA}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
    )
    try:
        from google.colab import userdata

        os.environ["KAGGLE_ACCESS_TOKEN"] = userdata.get("KAGGLE_ACCESS_TOKEN")
        print("Token de Kaggle cargado desde los secretos de Colab.")
    except Exception:
        print("Registre el secreto KAGGLE_ACCESS_TOKEN para poder descargar los datos.")

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("Entorno:", "Google Colab" if EN_COLAB else "local")
print("Raíz del proyecto:", RAIZ)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency

from src.data.make_dataset import asegurar_datos
from src.visualization.visualize import set_estilo

set_estilo()
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)

train = asegurar_datos()
print("Observaciones:", f"{train.shape[0]:,}", "| Variables:", train.shape[1])

## 1. Variables cuantitativas

Como se explicó en el cuaderno anterior, el conjunto original no trae ninguna variable numérica. Las cuatro que se analizan aquí fueron derivadas del texto de la explicación:

- `n_caracteres`: longitud del texto.
- `n_palabras`: cantidad de palabras.
- `long_media_palabra`: caracteres por palabra en promedio.
- `n_digitos`: cantidad de cifras numéricas.

### 1.1 Estadística descriptiva

In [ ]:
NUM = ["n_caracteres", "n_palabras", "long_media_palabra", "n_digitos"]

descriptiva = train[NUM].describe().T
descriptiva["rango_ic"] = descriptiva["75%"] - descriptiva["25%"]
descriptiva["cv"] = (descriptiva["std"] / descriptiva["mean"]).round(2)
descriptiva["asimetria"] = train[NUM].skew().round(2)
descriptiva["curtosis"] = train[NUM].kurt().round(2)
descriptiva.round(2)

### ¿Qué se observa?

Las cuatro variables comparten un mismo perfil: **asimetría positiva** y **curtosis alta**. Esto quiere decir que la mayoría de los valores se concentran en la parte baja y que existe una cola larga hacia la derecha.

El caso más claro es `n_palabras`. La mediana es de 13 palabras y el tercer cuartil de 19, pero el máximo llega a 177. La explicación típica es una sola frase corta, mientras que unas pocas son textos extensos.

`long_media_palabra` es la excepción parcial. Su coeficiente de variación es de apenas 0.15, la más estable de las cuatro, lo que es esperable porque la longitud media de las palabras en inglés varía poco entre textos.

`n_digitos` tiene el coeficiente de variación más alto, cercano a 1. Hay explicaciones sin ninguna cifra y otras con más de cien. Esta variable separa dos formas muy distintas de responder: describir con palabras o mostrar el cálculo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, col in zip(axes.ravel(), NUM):
    sns.histplot(train[col], bins=50, ax=ax, color="#4C72B0", edgecolor=None)
    ax.axvline(train[col].mean(), color="#C44E52", ls="--", lw=1.4, label="media")
    ax.axvline(train[col].median(), color="#55A868", ls="-", lw=1.4, label="mediana")
    ax.set_title(col)
    ax.set_ylabel("frecuencia")
    ax.legend()
fig.suptitle("Distribución de las variables derivadas del texto", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

Los histogramas confirman lo anterior. En las tres primeras variables la media queda a la derecha de la mediana (distribución sesgada hacia la derecha). En `n_digitos` se aprecia además un pico pronunciado en cero, correspondiente a las explicaciones puramente verbales.

### 1.2 Valores atípicos

Se identifican los valores atípicos con el criterio del rango intercuartílico, que marca como extremo todo lo que quede a más de 1.5 rangos intercuartílicos de los cuartiles.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 4))
for ax, col in zip(axes, NUM):
    sns.boxplot(y=train[col], ax=ax, color="#4C72B0", width=0.45, fliersize=1.5)
    ax.set_title(col)
    ax.set_ylabel("")
fig.suptitle("Diagramas de caja y bigotes", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
filas = []
for col in NUM:
    q1, q3 = train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    inf, sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    bajos = (train[col] < inf).sum()
    altos = (train[col] > sup).sum()
    filas.append({
        "variable": col, "limite_inferior": round(inf, 2), "limite_superior": round(sup, 2),
        "atipicos_bajos": bajos, "atipicos_altos": altos,
        "pct_atipicos": round((bajos + altos) / len(train) * 100, 2),
    })
pd.DataFrame(filas)

In [ ]:
print("Las cinco explicaciones más cortas:")
print(train.nsmallest(5, "n_caracteres")[["explicacion_limpia", "Category"]].to_string(index=False))
print()
print("La explicación más larga (primeros 260 caracteres):")
print(train.loc[train["n_caracteres"].idxmax(), "explicacion_limpia"][:260])
print()
print("Categoría de las 1,556 explicaciones más largas, en porcentaje:")
largas = train[train["n_caracteres"] > 150]
print((largas["Category"].value_counts(normalize=True) * 100).round(1).to_string())
print()
print("Categoría del total, en porcentaje, para comparar:")
print((train["Category"].value_counts(normalize=True) * 100).round(1).to_string())

### ¿Qué se observa y qué se decide?

Los valores atípicos representan entre el 1.6% y el 4.2% de las observaciones según la variable. Al inspeccionarlos se comprueba que **no son errores de registro sino respuestas reales y significativas**.

- En el extremo bajo aparecen respuestas como `d`, `??`, `-3` o `3/9`. Son estudiantes que escribieron la letra de la opción, un signo de duda o el resultado sin justificar nada. Casi todas caen en la categoría `Neither`, que es exactamente lo que esa categoría describe.
- En el extremo alto hay razonamientos largos y elaborados, con errores de ortografía pero con contenido. Al comparar su composición con la del conjunto completo se ve que las explicaciones largas están **sobrerrepresentadas en las categorías `Correct`** y subrepresentadas en `Neither`.

**Decisión:** los valores atípicos **no se eliminan ni se recortan**. Son informativos y su eliminación borraría precisamente la señal que interesa. Para los análisis que lo requieran se usan estadísticos robustos, como la mediana, en lugar de la media.

### 1.3 Relación entre las variables cuantitativas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

corr_p = train[NUM].corr()
sns.heatmap(corr_p, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, ax=axes[0], cbar_kws={"shrink": 0.8})
axes[0].set_title("Correlación de Pearson")

corr_s = train[NUM].corr(method="spearman")
sns.heatmap(corr_s, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, ax=axes[1], cbar_kws={"shrink": 0.8})
axes[1].set_title("Correlación de Spearman")
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

axes[0].hexbin(train["n_palabras"], train["n_caracteres"], gridsize=45, cmap="Blues", mincnt=1)
axes[0].set_xlabel("n_palabras"); axes[0].set_ylabel("n_caracteres")
axes[0].set_title("n_palabras frente a n_caracteres")

muestra = train.sample(4000, random_state=42)
sns.scatterplot(data=muestra, x="n_palabras", y="n_digitos", hue="tipo_explicacion",
                alpha=0.45, s=14, ax=axes[1])
axes[1].set_title("n_palabras frente a n_digitos, por tipo de explicación")
axes[1].legend(title="tipo_explicacion", loc="upper right")
fig.tight_layout()
plt.show()

### ¿Qué se observa?

La correlación entre `n_caracteres` y `n_palabras` es de **0.97**, prácticamente perfecta. Esto era esperable, pues las dos miden lo mismo, el largo del texto. Para el modelado conviene conservar solo una de las dos, porque la información que aportan es redundante.

El resto de correlaciones son moderadas y aportan información distinta.

- `n_palabras` y `n_digitos` correlacionan 0.48. Los textos más largos tienden a incluir más cálculo, pero la relación está lejos de ser determinista.
- `long_media_palabra` y `n_digitos` correlacionan **negativamente**, en torno a menos 0.26. Esto tiene sentido, pues las cifras son cadenas cortas, de modo que una explicación cargada de números baja el promedio de caracteres por palabra. Esta variable capta entonces algo que las otras no: el equilibrio entre escribir y calcular.

El gráfico de dispersión muestra además que las explicaciones del tipo `Neither` se concentran en la esquina inferior izquierda, es decir, son cortas y con pocas cifras. Es la primera evidencia visual de que las variables derivadas del texto sí separan, al menos parcialmente, las categorías.

## 2. Variables categóricas

### 2.1 La variable `Category`

Es la variable objetivo principal. Combina dos decisiones: si la opción elegida era correcta y qué revela la explicación.

In [ ]:
tabla = train["Category"].value_counts().to_frame("frecuencia")
tabla["proporcion"] = (tabla["frecuencia"] / len(train) * 100).round(2)
tabla["acumulado"] = tabla["proporcion"].cumsum().round(2)
tabla

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

orden = train["Category"].value_counts().index
sns.countplot(data=train, y="Category", order=orden, ax=axes[0], color="#4C72B0")
for cont in axes[0].containers:
    axes[0].bar_label(cont, fmt="%d", padding=3, fontsize=9)
axes[0].set_title("Frecuencia de cada categoría")
axes[0].set_xlabel("observaciones"); axes[0].set_ylabel("")
axes[0].set_xlim(0, 17000)

partes = train["tipo_explicacion"].value_counts()
axes[1].pie(partes, labels=partes.index, autopct="%1.1f%%", startangle=90,
            colors=sns.color_palette("deep", 3), wedgeprops={"edgecolor": "white"})
axes[1].set_title("Composición según lo que revela la explicación")
fig.tight_layout()
plt.show()

### ¿Qué se observa?

La distribución está claramente desbalanceada:

- `True_Correct` concentra el **40.34%** de las observaciones. Es el caso normal, el estudiante acierta y explica bien.
- `False_Misconception` aporta el **25.77%**. Es el caso central de la competencia, el estudiante falla y su explicación revela por qué.
- Las dos categorías menos frecuentes son de las más interesantes desde el punto de vista educativo. `True_Misconception` reúne apenas **403 casos, el 1.10%**, y `False_Correct` solo **227 casos, el 0.62%**.

`True_Misconception` merece atención especial. Corresponde al estudiante que marca la opción correcta pero razona mal. Es el caso que ningún sistema basado en el resultado puede detectar, y es también el más escaso. Un modelo entrenado sin cuidado simplemente lo ignorará, porque equivocarse en 403 casos de 36,696 casi no afecta la exactitud global.

### 2.2 La variable `Misconception`

In [ ]:
con_error = train[train["misconception"] != "NA"]
frec = con_error["misconception"].value_counts()

tabla_m = frec.to_frame("frecuencia")
tabla_m["pct_sobre_errores"] = (tabla_m["frecuencia"] / len(con_error) * 100).round(2)
tabla_m["pct_sobre_total"] = (tabla_m["frecuencia"] / len(train) * 100).round(2)
tabla_m["acumulado"] = tabla_m["pct_sobre_errores"].cumsum().round(2)

print("Observaciones con error conceptual:", f"{len(con_error):,}",
      f"({len(con_error) / len(train) * 100:.2f} % del total)")
print("Etiquetas distintas:", frec.size)
print("Razón entre la más y la menos frecuente:", round(frec.iloc[0] / frec.iloc[-1], 1), "a 1")
tabla_m.head(12)

In [ ]:
print("Las ocho etiquetas menos frecuentes:")
tabla_m.tail(8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

top = frec.head(20)
sns.barplot(x=top.values, y=top.index, ax=axes[0], color="#4C72B0")
axes[0].set_title("Las 20 etiquetas de error conceptual más frecuentes")
axes[0].set_xlabel("observaciones"); axes[0].set_ylabel("")

acum = (frec.cumsum() / frec.sum() * 100).values
axes[1].plot(range(1, len(acum) + 1), acum, marker="o", ms=4, color="#4C72B0")
axes[1].axhline(80, color="#C44E52", ls="--", lw=1.3, label="80 % de los casos")
axes[1].axhline(50, color="#55A868", ls="--", lw=1.3, label="50 % de los casos")
axes[1].set_xlabel("número de etiquetas, ordenadas por frecuencia")
axes[1].set_ylabel("porcentaje acumulado")
axes[1].set_title("Concentración de los errores conceptuales")
axes[1].legend()
fig.tight_layout()
plt.show()

for umbral in (50, 80, 90):
    n = int((acum < umbral).sum()) + 1
    print(f"Se necesitan {n:2d} de las 35 etiquetas para cubrir el {umbral} % de los casos.")

### ¿Qué se observa?

El desbalance es severo. Las 9,860 observaciones con error conceptual se reparten entre 35 etiquetas, pero de forma muy desigual: **la etiqueta más frecuente tiene 242 veces más casos que la menos frecuente**.

La curva de concentración lo resume bien: **siete etiquetas bastan para cubrir la mitad de los casos** y **quince cubren el 80%**. Las veinte restantes se reparten el quinto final. Hay etiquetas con menos de diez observaciones en todo el conjunto, como `Wrong_Operation` con seis.

Esto tiene una consecuencia práctica inmediata. Con seis ejemplos no se puede entrenar ni validar de forma fiable una clase. Las opciones son agrupar las etiquetas de cola larga, tratarlas con técnicas específicas para clases raras, o aceptar que el modelo no las aprenderá.

### 2.3 La variable objetivo completa

In [ ]:
objetivo = train["objetivo"].value_counts()
print("Combinaciones Category:Misconception distintas:", objetivo.size)
print("Frecuencia máxima:", f"{objetivo.max():,}", "| mínima:", objetivo.min())
print("Combinaciones con menos de 50 observaciones:", int((objetivo < 50).sum()))
print("Combinaciones con menos de 10 observaciones:", int((objetivo < 10).sum()))
print()
print("Las seis más frecuentes:")
print(objetivo.head(6).to_string())
print()
print("Las seis menos frecuentes:")
print(objetivo.tail(6).to_string())

### ¿Qué se observa?

El problema real no tiene 6 clases sino **65**. De ellas, **32 tienen menos de 50 observaciones** y **24 tienen menos de 10**. Una combinación aparece una sola vez en todo el conjunto.

Esto explica por qué la competencia evalúa con una métrica que admite tres predicciones. Con 65 clases tan desiguales, exigir una única respuesta correcta sería excesivamente estricto.

### 2.4 Las preguntas y las opciones de respuesta

In [ ]:
por_pregunta = (
    train.groupby("QuestionId")
    .agg(observaciones=("row_id", "size"),
         tasa_acierto=("respuesta_correcta", "mean"),
         errores_distintos=("misconception", lambda s: s[s != "NA"].nunique()))
    .assign(tasa_acierto=lambda d: (d["tasa_acierto"] * 100).round(1))
    .sort_values("observaciones", ascending=False)
)
por_pregunta

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

datos = por_pregunta.sort_values("observaciones")
axes[0].barh(datos.index.astype(str), datos["observaciones"], color="#4C72B0")
axes[0].set_title("Observaciones por pregunta")
axes[0].set_xlabel("observaciones"); axes[0].set_ylabel("QuestionId")

datos2 = por_pregunta.sort_values("tasa_acierto")
axes[1].barh(datos2.index.astype(str), datos2["tasa_acierto"], color="#DD8452")
axes[1].axvline(train["respuesta_correcta"].mean() * 100, color="#C44E52", ls="--",
                lw=1.4, label="promedio general")
axes[1].set_title("Porcentaje de respuestas correctas por pregunta")
axes[1].set_xlabel("% de aciertos"); axes[1].set_ylabel("QuestionId")
axes[1].legend()
fig.tight_layout()
plt.show()

### ¿Qué se observa?

Las 36,696 observaciones provienen de **solo 15 preguntas**. El número de respuestas por pregunta va de 673 a 4,857.

El conjunto no es una muestra amplia de problemas de matemática, es una muestra muy profunda de quince problemas concretos. La consecuencia se desarrolla en la sección 3.3.

La dificultad varía mucho entre preguntas. La pregunta 33474 la acierta solo el 35.7% de los estudiantes, mientras que la 32829 la acierta el 79.6%. Esa diferencia de más de cuarenta puntos indica que el conjunto cubre un rango amplio de dificultad.

## 3. Cruces entre variables

### 3.1 Corrección de la respuesta frente al tipo de explicación

Este cruce descompone la variable objetivo en sus dos dimensiones y permite ver si son independientes.

In [ ]:
contingencia = pd.crosstab(train["tipo_explicacion"], train["respuesta_correcta"], margins=True)
print("Frecuencias absolutas:")
print(contingencia.to_string())
print()
print("Proporciones por fila, en porcentaje:")
print((pd.crosstab(train["tipo_explicacion"], train["respuesta_correcta"],
                   normalize="index") * 100).round(2).to_string())

In [ ]:
prop = pd.crosstab(train["tipo_explicacion"], train["respuesta_correcta"], normalize="index") * 100

fig, ax = plt.subplots(figsize=(7.5, 4))
sns.heatmap(prop, annot=True, fmt=".1f", cmap="Blues", ax=ax, cbar_kws={"label": "% de la fila"})
ax.set_title("Tipo de explicación frente a corrección de la respuesta")
ax.set_xlabel("respuesta correcta"); ax.set_ylabel("tipo de explicación")
fig.tight_layout()
plt.show()

### ¿Qué se observa?

Las dos dimensiones están fuertemente relacionadas, pero no son la misma cosa.

- Cuando la explicación es correcta, el estudiante acertó el **98.49%** de las veces.
- Cuando la explicación revela un error conceptual, falló el **95.91%** de las veces.
- Cuando la explicación no es ni una cosa ni la otra, el reparto es **casi 50 y 50**, con un 55.41% de fallos.

La fila `Neither` revela que cuando el estudiante no llega a explicar su razonamiento, saber si acertó no dice nada sobre su comprensión. Puede haber acertado por suerte o por haber entendido sin saber expresarlo. Esta es exactamente la situación en la que la explicación escrita aporta información que la opción marcada no aporta.

Las dos celdas pequeñas son las difíciles: 403 casos de acierto con razonamiento equivocado y 227 casos de fallo con razonamiento correcto, probablemente errores de cálculo sobre una idea bien entendida.

### 3.2 Las variables del texto frente a la categoría

In [ ]:
resumen_cat = train.groupby("Category").agg(
    n=("row_id", "size"),
    palabras_mediana=("n_palabras", "median"),
    palabras_media=("n_palabras", "mean"),
    caracteres_mediana=("n_caracteres", "median"),
    digitos_mediana=("n_digitos", "median"),
).round(2).sort_values("palabras_mediana")
resumen_cat

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

orden = train.groupby("Category")["n_palabras"].median().sort_values().index
sns.boxplot(data=train, x="n_palabras", y="Category", order=orden, ax=axes[0],
            showfliers=False, color="#4C72B0")
axes[0].set_title("Número de palabras por categoría")
axes[0].set_xlabel("palabras"); axes[0].set_ylabel("")

sns.violinplot(data=train, x="n_digitos", y="Category", order=orden, ax=axes[1],
               cut=0, density_norm="width", color="#DD8452")
axes[1].set_xlim(0, 25)
axes[1].set_title("Cantidad de cifras por categoría")
axes[1].set_xlabel("dígitos"); axes[1].set_ylabel("")
fig.tight_layout()
plt.show()

In [ ]:
LOGICAS = ["tiene_digitos", "tiene_fraccion", "tiene_simbolo_mat", "tiene_porque"]
props = (train.groupby("Category")[LOGICAS].mean() * 100).round(1)
print(props.to_string())

fig, ax = plt.subplots(figsize=(11, 4.6))
props.plot(kind="bar", ax=ax, width=0.8)
ax.set_title("Presencia de rasgos del texto según la categoría")
ax.set_ylabel("% de explicaciones"); ax.set_xlabel("")
ax.tick_params(axis="x", rotation=25)
ax.legend(title="", bbox_to_anchor=(1.01, 1), loc="upper left")
fig.tight_layout()
plt.show()

### ¿Qué se observa?

Las variables derivadas del texto **sí discriminan**, aunque de forma parcial.

El patrón más nítido es el de la longitud. Las dos categorías `Neither` tienen una mediana de **11 palabras**, mientras que las categorías `Correct` llegan a **16 y 17 palabras**. La diferencia es consistente: **quien entiende, escribe más**.

La cantidad de cifras separa todavía mejor. La mediana de `False_Neither` es de **1 dígito** y la de `False_Correct` de **8**. Una explicación sin números suele ser una explicación sin razonamiento.

El gráfico de rasgos añade observaciones interesantes:

- `tiene_digitos` separa con fuerza. Aparece en más del 90% de las categorías `Correct` y en solo el 55% de `False_Neither`.
- `tiene_simbolo_mat` alcanza su valor más alto, **45.4%**, precisamente en `True_Misconception`, la categoría más escasa. El estudiante que acierta razonando mal tiende a mostrar operaciones escritas con símbolos.
- `tiene_porque` es sorprendentemente poco discriminante, entre 46% y 60% en todas las categorías. Que el estudiante escriba la palabra "because" no dice casi nada sobre si lo que sigue es correcto.

### 3.3 Dependencia entre las etiquetas y la pregunta

Esta es la pregunta central del análisis. La competencia busca errores conceptuales que **generalicen entre problemas distintos**. Se mide cuánto depende cada etiqueta de la pregunta concreta en la que apareció.

Para variables categóricas se usa la **V de Cramér**, que resume la asociación en una escala de 0 a 1, donde 0 indica independencia total y 1 indica que conocer una variable determina la otra.

In [ ]:
def v_de_cramer(x, y):
    '''Calcula la V de Cramér entre dos variables categóricas.'''
    tabla = pd.crosstab(x, y)
    chi2 = chi2_contingency(tabla, correction=False)[0]
    n = tabla.values.sum()
    filas, columnas = tabla.shape
    return np.sqrt((chi2 / n) / min(filas - 1, columnas - 1))


pares = [
    ("misconception ~ QuestionId", con_error["misconception"], con_error["QuestionId"]),
    ("misconception ~ MC_Answer", con_error["misconception"], con_error["MC_Answer"]),
    ("objetivo ~ QuestionId", train["objetivo"], train["QuestionId"]),
    ("Category ~ MC_Answer", train["Category"], train["MC_Answer"]),
    ("Category ~ QuestionId", train["Category"], train["QuestionId"]),
    ("tipo_explicacion ~ respuesta_correcta", train["tipo_explicacion"], train["respuesta_correcta"]),
]
asociacion = pd.DataFrame(
    [{"par": nombre, "v_cramer": round(v_de_cramer(a, b), 3)} for nombre, a, b in pares]
).sort_values("v_cramer", ascending=False)
asociacion

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colores = ["#C44E52" if v > 0.7 else "#4C72B0" for v in asociacion["v_cramer"]]
ax.barh(asociacion["par"], asociacion["v_cramer"], color=colores)
ax.bar_label(ax.containers[0], fmt="%.3f", padding=3)
ax.set_xlim(0, 1.08)
ax.axvline(0.7, color="#C44E52", ls="--", lw=1.2)
ax.set_title("Asociación entre variables categóricas (V de Cramér)")
ax.set_xlabel("V de Cramér")
ax.invert_yaxis()
fig.tight_layout()
plt.show()

In [ ]:
preguntas_por_error = con_error.groupby("misconception")["QuestionId"].nunique()

print("Distribución del número de preguntas en que aparece cada error conceptual:")
print(preguntas_por_error.value_counts().sort_index()
      .rename_axis("preguntas_distintas").to_frame("etiquetas").to_string())
print()
atadas = preguntas_por_error[preguntas_por_error == 1].index
print("Etiquetas que aparecen en una sola pregunta:", len(atadas), "de", len(preguntas_por_error))
print("Porcentaje de las observaciones con error conceptual que cubren:",
      round(con_error["misconception"].isin(atadas).mean() * 100, 2), "%")
print()
print("Las únicas etiquetas que aparecen en más de una pregunta:")
print(preguntas_por_error[preguntas_por_error > 1].to_string())

In [ ]:
unificadas = con_error["misconception"].str.lower()
preguntas_unif = con_error.assign(m=unificadas).groupby("m")["QuestionId"].nunique()

print("Efecto de unificar `Wrong_fraction` y `Wrong_Fraction`:")
print("  etiquetas antes:", preguntas_por_error.size, "| después:", preguntas_unif.size)
print("  atadas a una sola pregunta antes:", int((preguntas_por_error == 1).sum()),
      "| después:", int((preguntas_unif == 1).sum()))
print("  cobertura de las atadas a una pregunta antes:",
      round(con_error["misconception"].isin(atadas).mean() * 100, 2), "%")
atadas_unif = preguntas_unif[preguntas_unif == 1].index
print("  cobertura después:", round(unificadas.isin(atadas_unif).mean() * 100, 2), "%")

La corrección de la inconsistencia detectada en el cuaderno anterior mejora ligeramente el panorama, pero no lo cambia. Se pasa de 32 etiquetas atadas a una sola pregunta a 30, y la cobertura de esas etiquetas baja del 77.18% al 70.17%. Sigue siendo la gran mayoría de los casos.

In [ ]:
top15 = frec.head(15).index
mapa = pd.crosstab(con_error[con_error["misconception"].isin(top15)]["misconception"],
                   con_error[con_error["misconception"].isin(top15)]["QuestionId"])
mapa = mapa.loc[top15]

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.heatmap(mapa, cmap="Blues", annot=True, fmt="d", linewidths=0.5, linecolor="white",
            ax=ax, cbar_kws={"label": "observaciones"})
ax.set_title("En qué preguntas aparece cada error conceptual (15 más frecuentes)")
ax.set_xlabel("QuestionId"); ax.set_ylabel("")
fig.tight_layout()
plt.show()

### ¿Qué se observa?

La asociación entre `misconception` y `QuestionId` es de **0.984**, es decir, casi determinista. **Conocer la pregunta prácticamente determina qué errores conceptuales pueden aparecer.**

El mapa de calor lo muestra de forma visual: cada fila tiene sus observaciones concentradas en una sola columna. La matriz es casi diagonal en vez de estar poblada.

El conteo lo confirma: **32 de las 35 etiquetas aparecen en una única pregunta**, y esas 32 cubren el **77.18%** de las observaciones con error conceptual. Solo tres etiquetas (`Incomplete`, `Irrelevant` y `WNB`) aparecen en más de una pregunta, y ninguna en más de tres.

**Esta es la dificultad central del problema.** Un modelo entrenado sobre estos datos puede alcanzar buenos resultados simplemente aprendiendo a reconocer la pregunta y devolver el error conceptual más frecuente en ella, sin entender nada del razonamiento del estudiante. Ese modelo fracasará ante preguntas nuevas.

Conviene notar el contraste: `Category ~ QuestionId` da solo **0.169**. La categoría general no depende de la pregunta, pero la etiqueta específica sí. Dicho de otro modo, **decidir si hay un error conceptual es un problema generalizable; decir cuál es, con estos datos, no lo es**.

De aquí se desprende una decisión metodológica importante: **la validación debe agrupar por pregunta**, dejando preguntas completas fuera del entrenamiento. Una división aleatoria daría una estimación optimista y engañosa.

### 3.4 Explicaciones repetidas y ruido de las etiquetas

In [ ]:
clave = ["QuestionId", "MC_Answer", "explicacion_limpia"]
grupos = train.groupby(clave)["objetivo"].agg(["nunique", "size"])
repetidos = grupos[grupos["size"] > 1]

print("Combinaciones idénticas que aparecen más de una vez:", len(repetidos))
print("De ellas, con más de una etiqueta distinta:", int((repetidos["nunique"] > 1).sum()))
print("Porcentaje de desacuerdo entre las repetidas:",
      round((repetidos["nunique"] > 1).mean() * 100, 2), "%")
print()
print("Las explicaciones repetidas más comunes:")
print(train["explicacion_min"].value_counts().head(8).to_string())

### ¿Qué se observa?

De las 629 combinaciones que se repiten, **63 recibieron más de una etiqueta**, un **10.02%** de desacuerdo. Es una medida directa del ruido de anotación, y funciona como un techo práctico (ante entradas idénticas, ningún modelo puede acertar en ambos casos).

El conteo difiere levemente del que reportó el cuaderno anterior, 61 grupos, porque allí se usó el texto original y aquí el texto con los espacios normalizados. La normalización hace que dos explicaciones que solo diferían en el espaciado cuenten como la misma, de modo que se detectan dos desacuerdos adicionales.

Las explicaciones repetidas más comunes son frases genéricas y muy cortas, del tipo que se espera encontrar en la categoría `Neither`.

## 4. Señal léxica del texto

Las secciones anteriores midieron el texto, pero no lo leyeron. Ahora se busca **qué palabras** distinguen a cada categoría.

El procedimiento tiene dos pasos. Primero se representa cada explicación con **TF-IDF**, que pondera cada palabra por su frecuencia en el texto y la penaliza si aparece en muchos textos, de modo que quedan realzadas las palabras que distinguen. Se incluyen también los pares de palabras contiguas, porque expresiones como "not shaded" significan algo que sus palabras sueltas no significan.

Después se aplica la prueba **chi cuadrado**, que mide si la aparición de un término y la pertenencia a una clase son independientes. Los términos con mayor valor son los que más separan.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import chi2

vectorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=10, sublinear_tf=True)
X = vectorizador.fit_transform(train["explicacion_min"])
terminos = np.array(vectorizador.get_feature_names_out())

print("Matriz de términos:", X.shape)
print("Densidad de la matriz:", f"{X.nnz / (X.shape[0] * X.shape[1]) * 100:.3f} %")

In [ ]:
def terminos_discriminantes(matriz, etiquetas, clase, n=12):
    '''Devuelve los términos más asociados a una clase según chi cuadrado.

    Se conservan solo los términos sobrerrepresentados en la clase, porque
    chi cuadrado también premia a los que están llamativamente ausentes.
    '''
    y = (etiquetas == clase).astype(int).values
    puntajes, _ = chi2(matriz, y)
    media_dentro = np.asarray(matriz[y == 1].mean(axis=0)).ravel()
    media_fuera = np.asarray(matriz[y == 0].mean(axis=0)).ravel()
    puntajes = np.where(media_dentro > media_fuera, puntajes, -1)
    mejores = np.argsort(puntajes)[::-1][:n]
    return terminos[mejores], puntajes[mejores]


for clase in sorted(train["tipo_explicacion"].unique()):
    palabras, _ = terminos_discriminantes(X, train["tipo_explicacion"], clase)
    print(f"{clase:15s}: {', '.join(palabras)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, clase in zip(axes, sorted(train["tipo_explicacion"].unique())):
    palabras, puntajes = terminos_discriminantes(X, train["tipo_explicacion"], clase, n=12)
    ax.barh(palabras[::-1], puntajes[::-1], color="#4C72B0")
    ax.set_title(f"tipo_explicacion = {clase}")
    ax.set_xlabel("estadístico chi cuadrado")
fig.suptitle("Términos que más distinguen a cada tipo de explicación",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

### ¿Qué se observa?

Los tres grupos de términos arrojan resultados muy distintos.

**`Correct`** se caracteriza por **números concretos**: `12`, `24`, `72`, `is 24`, `120`. No son palabras que expresen comprensión, son **los resultados correctos de las quince preguntas**. Un modelo que aprenda estos términos no aprende matemática, aprende la clave de respuestas. Es la confirmación léxica del problema detectado en la sección 3.3.

**`Misconception`** se apoya en el vocabulario del contenido: `not shaded`, `triangles`, `positive`, `are not`. Son palabras ligadas al enunciado de una pregunta concreta, con la excepción de `positive`, que sí remite a una idea matemática transferible.

**`Neither`** es el grupo más interesante. Sus términos son `worked it out`, `how to`, `explain`, `not sure`, `don know`, `am`, `my`. No hablan de matemática en absoluto, hablan del propio estudiante y de su incertidumbre. **Esta es la única señal claramente independiente de la pregunta**: quien no sabe explicar lo dice de la misma manera sin importar el problema.

In [ ]:
for clase in frec.head(6).index:
    palabras, _ = terminos_discriminantes(X[train["misconception"].values != "NA"],
                                          con_error["misconception"], clase, n=10)
    print(f"{clase:24s}: {', '.join(palabras)}")

### ¿Qué se observa?

Al bajar al nivel de los errores conceptuales específicos, la dependencia del enunciado se vuelve evidente.

- `Subtraction` se identifica por `robert`, `cake`, `sally`, `eats`, `ate`. Son **los nombres de los personajes del problema**, no rasgos del error de razonamiento.
- `Incomplete` se identifica por `shaded`, `triangles`, `not shaded`, que describen la figura de una pregunta concreta.
- `Additive` y `Duplication` se apoyan en las cifras del ejercicio, `10`, `15`, `2x5`.

La excepción es **`Positive`**, cuyos términos son `two negatives`, `make positive`, `negative`. Aquí sí aparece el enunciado de una regla matemática equivocada, expresada por el propio estudiante, y esa formulación sería reconocible en cualquier otro problema con números negativos. Es el ejemplo de lo que la competencia realmente busca.

In [ ]:
contador = CountVectorizer(ngram_range=(2, 2), min_df=20)
B = contador.fit_transform(train["explicacion_min"])
bigramas = np.array(contador.get_feature_names_out())

for clase in sorted(train["tipo_explicacion"].unique()):
    sub = B[(train["tipo_explicacion"] == clase).values]
    sumas = np.asarray(sub.sum(axis=0)).ravel()
    mejores = np.argsort(sumas)[::-1][:8]
    print(f"{clase:15s}: " + ", ".join(f"{bigramas[i]} ({sumas[i]})" for i in mejores))

### ¿Qué se observa?

Los pares de palabras más frecuentes son **prácticamente los mismos en las tres clases**: `think this`, `this because`, `it is`, `the answer`. Corresponden a la fórmula con la que casi todos los estudiantes abren su respuesta.

**Lo frecuente no es lo informativo.** Por eso la prueba chi cuadrado, que busca términos desigualmente repartidos, encuentra señal donde el simple conteo de frecuencias no encuentra nada.

### 4.1 Visualización del espacio de los textos

Se reduce la matriz TF-IDF a dos dimensiones para observar si las categorías forman grupos separados.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=0)
P = svd.fit_transform(X)
print("Varianza explicada por las dos componentes:", svd.explained_variance_ratio_.round(4),
      "| total:", round(svd.explained_variance_ratio_.sum() * 100, 2), "%")

idx = train.sample(6000, random_state=1).index
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, color_por, titulo in [
    (axes[0], "tipo_explicacion", "Coloreado por tipo de explicación"),
    (axes[1], "QuestionId", "Coloreado por pregunta"),
]:
    sns.scatterplot(x=P[idx, 0], y=P[idx, 1], hue=train.loc[idx, color_por].astype(str),
                    s=9, alpha=0.55, ax=ax, legend=(color_por == "tipo_explicacion"),
                    palette="deep" if color_por == "tipo_explicacion" else "tab20")
    ax.set_title(titulo)
    ax.set_xlabel("componente 1"); ax.set_ylabel("componente 2")
fig.suptitle("Proyección de las explicaciones en dos dimensiones",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

### ¿Qué se observa?

Las dos componentes explican apenas el **2.06%** de la varianza. Es un valor muy bajo, y por sí mismo ya es un resultado. El espacio de los textos es de dimensión alta y no se deja resumir en dos direcciones.

Al colorear por tipo de explicación, los puntos aparecen **completamente mezclados**. No hay grupos separados. Al colorear por pregunta, en cambio, se distinguen agrupamientos.

La lectura conjunta es coherente con todo lo anterior: **lo que más varía entre las explicaciones no es si contienen un error conceptual, sino de qué pregunta hablan**. El vocabulario del enunciado domina la representación, y la señal del razonamiento queda debajo.

## 5. Hallazgos y conclusiones

### 5.1 Resumen de los hallazgos

**Sobre la estructura y la calidad de los datos**

1. El conjunto tiene **36,696 observaciones y 7 variables**, de las cuales dos son de texto libre y ninguna es numérica. Las nueve variables cuantitativas usadas en el análisis fueron derivadas del texto.
2. No hay valores faltantes reales. El 73.13% de vacíos en `Misconception` es estructural y corresponde exactamente a las categorías sin error conceptual.
3. Las etiquetas contienen **ruido medible**: 63 combinaciones idénticas de pregunta, respuesta y explicación recibieron etiquetas distintas, un 10.02% de desacuerdo entre las repetidas, y una pregunta presenta 18 filas con una clave de respuesta inconsistente. Además, un mismo error conceptual figura con dos nombres distintos, `Wrong_fraction` y `Wrong_Fraction`, uno por cada pregunta en que fue anotado.

**Sobre la distribución de las clases**

4. La variable objetivo real tiene **65 combinaciones**, no 6. De ellas **24 tienen menos de 10 observaciones**.
5. El desbalance es severo. Siete de las 35 etiquetas de error conceptual cubren la mitad de los casos, quince cubren el 80%, y la más frecuente tiene **242 veces más casos que la menos frecuente**.
6. `True_Misconception`, el caso de acertar razonando mal, reúne solo **403 observaciones, el 1.10%**. Es el caso de mayor valor educativo y el peor representado.

**Sobre la capacidad de generalizar**

7. Todo el conjunto proviene de **solo 15 preguntas**.
8. La asociación entre el error conceptual y la pregunta es de **0.984** en la V de Cramér, casi determinista. **32 de las 35 etiquetas aparecen en una única pregunta** y cubren el 77.18% de los casos con error conceptual. Al unificar la etiqueta escrita de dos formas distintas, las cifras mejoran a 30 de 34 y a un 70.17% de cobertura, sin alterar la conclusión.
9. En contraste, la asociación entre `Category` y la pregunta es de solo **0.169**. Detectar *si* hay un error conceptual es generalizable; identificar *cuál* es, con estos datos, no lo es.

**Sobre la señal disponible en el texto**

10. Las variables derivadas del texto discriminan de forma parcial pero consistente. La mediana de palabras va de 11 en las categorías `Neither` a 17 en `False_Correct`, y la de dígitos de 1 a 8. **Quien comprende, escribe más y calcula más.**
11. `n_caracteres` y `n_palabras` correlacionan **0.97** y son redundantes. `long_media_palabra` aporta información distinta, al correlacionar negativamente con `n_digitos`.
12. Los términos que distinguen a la clase `Correct` son **los resultados numéricos de las preguntas**, y los que distinguen a errores como `Subtraction` son **los nombres de los personajes del enunciado**. La señal que un modelo captaría con facilidad es la equivocada.
13. La excepción son dos casos. La clase `Neither` se identifica por expresiones de incertidumbre (`worked it out`, `not sure`, `don know`) que son independientes del problema, y el error `Positive` se identifica por el enunciado explícito de una regla equivocada (`two negatives make positive`). **Ese es el tipo de señal que sí generaliza.**
14. La proyección en dos dimensiones explica solo el **2.06%** de la varianza y las clases aparecen mezcladas, mientras que las preguntas sí se agrupan. El vocabulario del enunciado domina la representación del texto.

### 5.2 Conclusiones

- **El problema tiene dos niveles de dificultad muy distinta.** Decidir la `Category`, con seis clases y baja dependencia de la pregunta, es una tarea abordable. Identificar la `Misconception` específica, con 35 clases casi determinadas por la pregunta, es mucho más difícil. Conviene tratarlas como dos etapas y no como una sola clasificación de 65 clases.

- **La validación debe agrupar por pregunta.** Dada la asociación de 0.984 entre etiqueta y pregunta, una división aleatoria produciría una estimación optimista y engañosa. El esquema correcto deja preguntas completas fuera del entrenamiento, de modo que la métrica mida lo que la competencia evalúa, que es el comportamiento ante problemas nunca vistos.

- **Hay que forzar al modelo a leer el razonamiento y no el enunciado.** Las estrategias razonables son restar del texto las palabras propias del enunciado, ponderar a la baja los nombres propios y las cifras específicas, y dar peso a las formulaciones de reglas. El objetivo es que el modelo se parezca al caso `Positive` y no al caso `Subtraction`.

- **El desbalance exige un tratamiento explícito.** Con 24 clases de menos de diez observaciones, la exactitud global es una métrica engañosa. Corresponde usar métricas macro o ponderadas, ponderar las clases en el entrenamiento y considerar agrupar las etiquetas de la cola larga. El formato de entrega, que admite tres predicciones, sugiere además trabajar con probabilidades ordenadas en lugar de una única respuesta.

- **Hay que fijar las expectativas realmente alcanzables.** El 10.02% de desacuerdo entre observaciones idénticas es un techo que ningún modelo puede superar. Las 403 observaciones de `True_Misconception` difícilmente alcanzan para aprender ese caso, aunque sea el de mayor valor educativo.

### 5.3 Siguientes pasos

1. Unificar las etiquetas escritas de más de una forma antes de entrenar, y revertir la unificación al generar las predicciones.
2. Construir un modelo de referencia sencillo, TF-IDF con regresión logística, validado con agrupación por pregunta, para establecer una línea base honesta.
3. Comparar esa línea base con la de un modelo que solo mire la pregunta y la opción elegida, ignorando el texto. La diferencia entre ambas mide cuánto aporta realmente la explicación.
4. Evaluar por separado la etapa de categoría y la de error conceptual específico.
5. Probar representaciones que reduzcan el peso del vocabulario del enunciado antes de pasar a modelos más complejos.